In [1]:
import torch
import torch.nn.functional as F

target = torch.tensor([0])

correct_logits = torch.tensor([[5.0, 1.0, 0.0]])
wrong_logits = torch.tensor([[0.0, 5.0, 1.0]])

print("正确答案的编号:", target.item())
print("答对且自信时的 loss:", F.cross_entropy(correct_logits, target).item())
print("答错且自信时的 loss:", F.cross_entropy(wrong_logits, target).item())

正确答案的编号: 0
答对且自信时的 loss: 0.024744924157857895
答错且自信时的 loss: 5.024744987487793


In [1]:
tokens = ["BOS", "你好", "，", "世界", "EOS"]

input_tokens = tokens[:-1]
label_tokens = tokens[1:]

print("模型看到的输入:", input_tokens)
print("每个位置的正确答案:", label_tokens)

for i in range(len(input_tokens)):
    print(
        "位置", i,
        "看到", input_tokens[i],
        "应该预测", label_tokens[i]
    )


模型看到的输入: ['BOS', '你好', '，', '世界']
每个位置的正确答案: ['你好', '，', '世界', 'EOS']
位置 0 看到 BOS 应该预测 你好
位置 1 看到 你好 应该预测 ，
位置 2 看到 ， 应该预测 世界
位置 3 看到 世界 应该预测 EOS


In [1]:
import torch

input_ids = torch.tensor([[1, 101, 102, 103, 2]])

# 假装模型在 5 个位置都输出了分数
logits = torch.zeros((1, 5, 10))

shift_logits = logits[:, :-1, :]
shift_labels = input_ids[:, 1:]

print("原 input_ids:", input_ids.tolist())
print("shift_logits shape:", tuple(shift_logits.shape))
print("shift_labels:", shift_labels.tolist())

原 input_ids: [[1, 101, 102, 103, 2]]
shift_logits shape: (1, 4, 10)
shift_labels: [[101, 102, 103, 2]]


In [3]:
print('123')

123


In [4]:
tokens = ["BOS", "你好", "，", "世界", "EOS"]

input_tokens = ["BOS", "你好", "，", "世界", "EOS", "PAD", "PAD"]

label_tokens = ["BOS", "你好", "，", "世界", "EOS", "-100", "-100"]

attention_mask = [1, 1, 1, 1, 1, 0, 0]

print("位置       0      1      2      3      4      5      6")
print("输入       ", input_tokens)
print("标签       ", label_tokens)
print("注意力开关 ", attention_mask)

位置       0      1      2      3      4      5      6
输入        ['BOS', '你好', '，', '世界', 'EOS', 'PAD', 'PAD']
标签        ['BOS', '你好', '，', '世界', 'EOS', '-100', '-100']
注意力开关  [1, 1, 1, 1, 1, 0, 0]


In [5]:
# 假设原文本已经被切成了 3 个 token
raw_tokens = [101, 102, 103]

bos_id = 1       # 开始标记
eos_id = 2       # 结束标记
pad_id = 0       # 补齐标记
max_length = 6   # 每条样本统一为 6 个位置

# 1. 加上开始和结束标记
tokens = [bos_id] + raw_tokens + [eos_id]

# 2. 右侧补 PAD，直到长度变成 max_length
input_ids = tokens + [pad_id] * (max_length - len(tokens))

# 3. 复制一份，作为训练答案
labels = input_ids.copy()

# 4. PAD 不参与评分，所以改成 -100
for i in range(len(labels)):
    if labels[i] == pad_id:
        labels[i] = -100

# 5. 真实内容标 1，PAD 标 0
attention_mask = []
for token_id in input_ids:
    if token_id == pad_id:
        attention_mask.append(0)
    else:
        attention_mask.append(1)

print("位置:           ", list(range(max_length)))
print("input_ids:      ", input_ids)
print("labels:         ", labels)
print("attention_mask: ", attention_mask)

位置:            [0, 1, 2, 3, 4, 5]
input_ids:       [1, 101, 102, 103, 2, 0]
labels:          [1, 101, 102, 103, 2, -100]
attention_mask:  [1, 1, 1, 1, 1, 0]


In [6]:
# 假设 tokenizer 已经把“你好，世界”切成了这 3 个 token id
raw_token_ids = [101, 102, 103]

bos_id = 1
eos_id = 2
pad_id = 0
max_length = 6

print("第 1 步，原始 token:", raw_token_ids)

# 加上开始和结束标记
tokens = [bos_id] + raw_token_ids + [eos_id]
print("第 2 步，加 BOS/EOS:", tokens)

# 右侧补 PAD
input_ids = tokens + [pad_id] * (max_length - len(tokens))
print("第 3 步，补 PAD:", input_ids)

# 复制 input_ids 作为 labels
labels = input_ids.copy()

# PAD 不参与 loss
for i in range(len(labels)):
    if labels[i] == pad_id:
        labels[i] = -100

print("第 4 步，labels:", labels)

# 真实 token 为 1，PAD 为 0
attention_mask = []

for token_id in input_ids:
    if token_id == pad_id:
        attention_mask.append(0)
    else:
        attention_mask.append(1)

print("第 5 步，attention_mask:", attention_mask)

第 1 步，原始 token: [101, 102, 103]
第 2 步，加 BOS/EOS: [1, 101, 102, 103, 2]
第 3 步，补 PAD: [1, 101, 102, 103, 2, 0]
第 4 步，labels: [1, 101, 102, 103, 2, -100]
第 5 步，attention_mask: [1, 1, 1, 1, 1, 0]


In [7]:
import torch
from torch.utils.data import DataLoader

samples = [
    (
        torch.tensor([1, 101, 102, 2, 0, 0]),
        torch.tensor([1, 101, 102, 2, -100, -100]),
        torch.tensor([1, 1, 1, 1, 0, 0]),
    ),
    (
        torch.tensor([1, 201, 202, 203, 2, 0]),
        torch.tensor([1, 201, 202, 203, 2, -100]),
        torch.tensor([1, 1, 1, 1, 1, 0]),
    ),
]

loader = DataLoader(samples, batch_size=2, shuffle=False)

batch = next(iter(loader))
input_ids, labels, attention_mask = batch

print("input_ids shape:", tuple(input_ids.shape))
print("labels shape:", tuple(labels.shape))
print("attention_mask shape:", tuple(attention_mask.shape))
print()
print("input_ids:")
print(input_ids)
print()
print("labels:")
print(labels)
print()
print("attention_mask:")
print(attention_mask)

input_ids shape: (2, 6)
labels shape: (2, 6)
attention_mask shape: (2, 6)

input_ids:
tensor([[  1, 101, 102,   2,   0,   0],
        [  1, 201, 202, 203,   2,   0]])

labels:
tensor([[   1,  101,  102,    2, -100, -100],
        [   1,  201,  202,  203,    2, -100]])

attention_mask:
tensor([[1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0]])


In [8]:
import torch

a = torch.tensor(7)
b = torch.tensor([1, 2, 3])
c = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
])
d = torch.zeros((2, 4, 10))

for name, value in [("a", a), ("b", b), ("c", c), ("d", d)]:
    print(name)
    print("内容:", value)
    print("形状:", tuple(value.shape))
    print("维度数:", value.ndim)
    print()

a
内容: tensor(7)
形状: ()
维度数: 0

b
内容: tensor([1, 2, 3])
形状: (3,)
维度数: 1

c
内容: tensor([[1, 2, 3],
        [4, 5, 6]])
形状: (2, 3)
维度数: 2

d
内容: tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])
形状: (2, 4, 10)
维度数: 3



In [9]:
import torch
from torch.utils.data import DataLoader

# 4 条样本，每条样本有 3 个字段
samples = [
    (
        torch.tensor([0, 10, 11]),  # input_ids
        torch.tensor([0, 10, 11]),  # labels
        torch.tensor([1, 1, 1]),     # attention_mask
    ),
    (
        torch.tensor([1, 20, 21]),
        torch.tensor([1, 20, 21]),
        torch.tensor([1, 1, 1]),
    ),
    (
        torch.tensor([2, 30, 31]),
        torch.tensor([2, 30, 31]),
        torch.tensor([1, 1, 1]),
    ),
    (
        torch.tensor([3, 40, 41]),
        torch.tensor([3, 40, 41]),
        torch.tensor([1, 1, 1]),
    ),
]

loader = DataLoader(
    samples,
    batch_size=2,
    shuffle=False,
)

for step, (input_ids, labels, attention_mask) in enumerate(loader, start=1):
    print("当前 step:", step)
    print("这一批的 input_ids:")
    print(input_ids)
    print("这一批的形状:", tuple(input_ids.shape))
    print()

当前 step: 1
这一批的 input_ids:
tensor([[ 0, 10, 11],
        [ 1, 20, 21]])
这一批的形状: (2, 3)

当前 step: 2
这一批的 input_ids:
tensor([[ 2, 30, 31],
        [ 3, 40, 41]])
这一批的形状: (2, 3)



In [10]:
import torch

sample_0 = torch.tensor([0, 10, 11])
sample_1 = torch.tensor([1, 20, 21])

batch = torch.stack([sample_0, sample_1])

print("第 0 条样本:", sample_0)
print("第 1 条样本:", sample_1)
print()
print("叠成一批后:")
print(batch)
print("形状:", tuple(batch.shape))

第 0 条样本: tensor([ 0, 10, 11])
第 1 条样本: tensor([ 1, 20, 21])

叠成一批后:
tensor([[ 0, 10, 11],
        [ 1, 20, 21]])
形状: (2, 3)


In [11]:
import torch

# 一批输入：2 条样本，每条 4 个位置
input_ids = torch.tensor([
    [1, 10, 11, 2],
    [1, 20, 21, 2],
])

labels = input_ids.clone()

attention_mask = torch.tensor([
    [1, 1, 1, 1],
    [1, 1, 1, 1],
])

batch_size = input_ids.shape[0]
seq_len = input_ids.shape[1]
vocab_size = 5

# 假装模型输出 logits
logits = torch.zeros((batch_size, seq_len, vocab_size))

print("input_ids 形状:", tuple(input_ids.shape))
print("labels 形状:", tuple(labels.shape))
print("attention_mask 形状:", tuple(attention_mask.shape))
print("logits 形状:", tuple(logits.shape))

input_ids 形状: (2, 4)
labels 形状: (2, 4)
attention_mask 形状: (2, 4)
logits 形状: (2, 4, 5)


In [12]:
import torch
import torch.nn.functional as F

# DataLoader 交给模型的一批数据：2 条样本，每条 4 个位置
input_ids = torch.tensor([
    [1, 3, 4, 2],
    [1, 2, 0, 0],
])

labels = input_ids.clone()
labels[labels == 0] = -100

attention_mask = torch.tensor([
    [1, 1, 1, 1],
    [1, 1, 0, 0],
])

# 假装模型已经计算出了 logits。
# 2 条样本 × 4 个位置 × 5 个候选 token 分数
logits = torch.randn((2, 4, 5))

# 预测“下一个 token”，所以两边错开一位
shift_logits = logits[:, :-1, :]
shift_labels = labels[:, 1:]

# 把所有可预测位置放在一起，计算一个总 loss
loss = F.cross_entropy(
    shift_logits.reshape(-1, 5),
    shift_labels.reshape(-1),
    ignore_index=-100,
)

print("input_ids:       ", tuple(input_ids.shape))
print("labels:          ", tuple(labels.shape))
print("attention_mask:  ", tuple(attention_mask.shape))
print("logits:          ", tuple(logits.shape))
print("shift_logits:    ", tuple(shift_logits.shape))
print("shift_labels:    ", tuple(shift_labels.shape))
print("loss:            ", round(loss.item(), 4))

input_ids:        (2, 4)
labels:           (2, 4)
attention_mask:   (2, 4)
logits:           (2, 4, 5)
shift_logits:     (2, 3, 5)
shift_labels:     (2, 3)
loss:             1.4454


In [13]:
import torch
import torch.nn.functional as F

logits = torch.tensor([[2.0, 0.0]])

for label in [0, 1]:
    target = torch.tensor([label])

    probabilities = torch.softmax(logits, dim=-1)
    loss = F.cross_entropy(logits, target)

    print("label:", label)
    print("概率:", probabilities.tolist())
    print("loss:", round(loss.item(), 3))
    print()

label: 0
概率: [[0.8807970285415649, 0.11920291185379028]]
loss: 0.127

label: 1
概率: [[0.8807970285415649, 0.11920291185379028]]
loss: 2.127

